# Problem B: Duplicate and anomalous transaction checker

**ENAI11-100 Thinking With AI, Skills Assignment 1.** Harbour Lane Café transactions, 7–9 September 2026.

This notebook follows the Problem B pseudocode (section 2.1.2 of the report) and flowcharts (Figures 4–9).
It reads `SA1_ProblemB_CustomerTransactions.xlsx`, flags duplicate or anomalous transactions with a reason
and a severity, and then lets the café owner explore and review the flags through a menu.

*AI disclosure: this code was generated with Claude (Anthropic) from my own pseudocode and design decisions; see the AI Use Log.*

**Stated assumptions (thresholds)**

| Rule | Assumption | Severity |
|---|---|---|
| Missing field | Any blank field is flagged; it does not stop the other checks | Customer ID blank = Review, any other field = Error |
| Not a number | Quantity, Unit price and Line total must be numbers; text such as 'two' is flagged and that value's checks are skipped | Error |
| Not a date | Timestamp must be a real date and time; text such as '7/9 morning' is flagged and that record is left out of the time-gap checks | Error |
| Quantity | Normal range is 1–5; 0 or negative is invalid | Below 1 = Error, above 5 = Review |
| Item | Must be on the Price List (capitals and start/end spaces ignored) | Error |
| Price | Unit price must equal the list price exactly, to the cent | Error |
| Line total | Line total must equal Quantity × Unit price, to the cent | Error |
| Suspected duplicate | Same customer and same item, 60 seconds or less after the previous one (capitals and start/end spaces ignored). A record with a blank Customer ID cannot be matched, so it is never a suspected duplicate | Error (later record only) |
| Repeated transaction ID | Transaction IDs should be unique (workbook README), so any ID used more than once is flagged, however far apart (capitals and start/end spaces ignored). 60 seconds or less apart looks like a double entry; further apart looks like a mistyped ID | 60 s or less = Error, more than 60 s = Review (later record only) |

## 1. Settings and imports

In [1]:
import pandas as pd

# ===== SETTINGS (stated assumptions) =====
DATA_FILE       = "SA1_ProblemB_CustomerTransactions.xlsx"   # keep the workbook in the same folder as this notebook
OUTPUT_FILE     = "ProblemB_Flags.xlsx"
MAX_GAP_SECONDS = 60     # repeats 60 s apart or less are suspicious
MIN_QTY         = 1      # 0 or negative = invalid
MAX_QTY         = 5      # above 5 = unusually high

# Column names exactly as they appear in the 'Transactions' sheet
COLUMNS = ["TransactionID", "Timestamp", "CustomerID", "Item",
           "Quantity", "UnitPrice (AUD)", "LineTotal (AUD)", "PaymentMethod"]

## 2. Read the input

In [2]:
def load_data(path):
    """Read the Transactions and Price List sheets.

    Adds a 'Row' column holding the spreadsheet row number (header = row 1),
    so two records that share a Transaction ID (e.g. T0038) can be told apart.
    """
    transactions = pd.read_excel(path, sheet_name="Transactions")
    transactions.insert(0, "Row", transactions.index + 2)

    price_list = pd.read_excel(path, sheet_name="Price List")
    # The sheet ends with a blank row and a note; keep only rows with an item and a price
    price_list = price_list.dropna(subset=["Item", "List Price (AUD)"])
    return transactions, price_list


try:
    transactions, price_list = load_data(DATA_FILE)
except FileNotFoundError:
    raise FileNotFoundError(f"Can't find {DATA_FILE}. Put the workbook in the same folder as this notebook, then run again.") from None
print(f"Loaded {len(transactions)} transactions and {len(price_list)} price-list items.")

Loaded 44 transactions and 12 price-list items.


## 3. Core checks (no typing needed, so the tests can run these directly)

In [3]:
def is_blank(value):
    """True for an empty cell, or text that is only spaces."""
    if pd.isna(value):
        return True
    return isinstance(value, str) and value.strip() == ""


def to_number(value):
    """The value as a number, or None if it is blank or not a number (e.g. 'two')."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return None if pd.isna(number) else number


def to_time(value):
    """The value as a date and time, or None if it is blank or not a date (e.g. '7/9 morning')."""
    try:
        time = pd.Timestamp(value)
    except (TypeError, ValueError):
        return None
    return None if pd.isna(time) else time


def normalise(text):
    """Remove start/end spaces and ignore capitals: 'Flat White ' -> 'flat white'."""
    return str(text).strip().lower()


def money(value):
    return f"${value:.2f}"


def describe_gap(seconds):
    """A time gap in words: 33 -> '33 s', 310 -> '5 min 10 s', 11374 -> '3 h 9 min'."""
    seconds = int(round(seconds))
    if seconds < 60:
        return f"{seconds} s"
    minutes, secs = divmod(seconds, 60)
    if minutes < 60:
        return f"{minutes} min {secs} s" if secs else f"{minutes} min"
    hours, minutes = divmod(minutes, 60)
    return f"{hours} h {minutes} min" if minutes else f"{hours} h"


def make_flag(row, rule, reason, severity):
    return {"Row": row["Row"], "TransactionID": row["TransactionID"],
            "CustomerID": row["CustomerID"], "Item": row["Item"],
            "Rule": rule, "Reason": reason, "Severity": severity}


NUMBER_FIELDS = ["Quantity", "UnitPrice (AUD)", "LineTotal (AUD)"]


def find_anomalies(transactions, price_list,
                   max_gap=MAX_GAP_SECONDS, min_qty=MIN_QTY, max_qty=MAX_QTY):
    """Run every check and return a DataFrame with one line per flag."""
    flags = []
    price_lookup = {normalise(item): price
                    for item, price in zip(price_list["Item"], price_list["List Price (AUD)"])}

    for _, row in transactions.iterrows():
        # Check 1 - missing fields (does NOT stop the other checks)
        for field in COLUMNS:
            if is_blank(row[field]):
                if field == "CustomerID":
                    flags.append(make_flag(row, "Missing field", "Customer ID is blank", "Review"))
                else:
                    flags.append(make_flag(row, "Missing field", f"{field} is blank", "Error"))

        # Check 1b - number fields that hold text are flagged (instead of crashing the checks below)
        for field in NUMBER_FIELDS:
            if not is_blank(row[field]) and to_number(row[field]) is None:
                flags.append(make_flag(row, "Not a number", f"{field} '{row[field]}' is not a number", "Error"))

        # Check 1c - a Timestamp that holds text is flagged (instead of crashing the time-gap checks)
        if not is_blank(row["Timestamp"]) and to_time(row["Timestamp"]) is None:
            flags.append(make_flag(row, "Not a date", f"Timestamp '{row['Timestamp']}' is not a date and time", "Error"))

        qty = to_number(row["Quantity"])
        price = to_number(row["UnitPrice (AUD)"])
        total = to_number(row["LineTotal (AUD)"])

        # Check 2 - quantity (needs: a numeric Quantity)
        if qty is not None:
            if qty < min_qty:
                flags.append(make_flag(row, "Invalid quantity",
                                       f"Quantity is {row['Quantity']}, must be {min_qty} or more", "Error"))
            elif qty > max_qty:
                flags.append(make_flag(row, "High quantity",
                                       f"Quantity {row['Quantity']} is above normal range {min_qty}-{max_qty}", "Review"))

        # Check 3 - item on price list (needs: Item), capitals and spaces ignored
        # Check 4 - unit price vs list price (needs: Item on list + a numeric UnitPrice)
        if not is_blank(row["Item"]):
            item_key = normalise(row["Item"])
            if item_key not in price_lookup:
                flags.append(make_flag(row, "Unknown item",
                                       f"'{row['Item']}' is not on the price list", "Error"))
            elif price is not None:
                listed = price_lookup[item_key]
                if round(price, 2) != round(listed, 2):
                    flags.append(make_flag(row, "Wrong price",
                                           f"Charged {money(price)}, list price {money(listed)}", "Error"))

        # Check 5 - line total vs quantity x unit price (needs: all three as numbers), to the cent
        if qty is not None and price is not None and total is not None:
            expected = qty * price
            if abs(expected - total) > 0.005:
                flags.append(make_flag(row, "Wrong line total",
                                       f"Line total {money(total)}, but {row['Quantity']} x {money(price)} = {money(expected)}",
                                       "Error"))

    find_close_repeats(transactions, ["CustomerID", "Item"], "Suspected duplicate", max_gap, flags)
    find_repeated_ids(transactions, max_gap, flags)

    columns = ["Row", "TransactionID", "CustomerID", "Item", "Rule", "Reason", "Severity"]
    return pd.DataFrame(flags, columns=columns).sort_values("Row", kind="stable").reset_index(drop=True)


def find_close_repeats(rows, key_fields, rule, max_gap, flags):
    """Flag the later record of any two records that share the key and are max_gap seconds apart or less."""
    groups = {}
    for _, row in rows.iterrows():
        if any(is_blank(row[f]) for f in key_fields) or to_time(row["Timestamp"]) is None:
            continue                                   # a blank Customer ID or bad Timestamp can't be matched
        key = tuple(normalise(row[f]) for f in key_fields)   # 'C1004 ' matches 'C1004'
        groups.setdefault(key, []).append(row)

    for group in groups.values():
        if len(group) < 2:
            continue
        # Earliest first; equal times keep spreadsheet order
        group.sort(key=lambda r: (to_time(r["Timestamp"]), r["Row"]))
        for previous, current in zip(group, group[1:]):
            gap = (to_time(current["Timestamp"]) - to_time(previous["Timestamp"])).total_seconds()
            if gap <= max_gap:
                flags.append(make_flag(current, rule,
                                       f"{describe_gap(gap)} after {previous['TransactionID']}", "Error"))


def find_repeated_ids(rows, max_gap, flags):
    """Flag every later use of a Transaction ID, however far apart.
    60 s or less apart = Error (looks like a double entry); further apart or time unknown = Review (looks like a mistyped ID)."""
    groups = {}
    for _, row in rows.iterrows():
        if not is_blank(row["TransactionID"]):
            groups.setdefault(normalise(row["TransactionID"]), []).append(row)

    for group in groups.values():
        if len(group) < 2:
            continue
        # Earliest first; a record with no usable time goes last; equal times keep spreadsheet order
        group.sort(key=lambda r: (to_time(r["Timestamp"]) or pd.Timestamp.max, r["Row"]))
        first = group[0]
        for current in group[1:]:
            start, end = to_time(first["Timestamp"]), to_time(current["Timestamp"])
            if start is None or end is None:
                flags.append(make_flag(current, "Repeated transaction ID",
                                       f"Same ID as row {first['Row']} (time unknown)", "Review"))
                continue
            gap = (end - start).total_seconds()
            severity = "Error" if gap <= max_gap else "Review"
            flags.append(make_flag(current, "Repeated transaction ID",
                                   f"Same ID as row {first['Row']}, {describe_gap(gap)} earlier", severity))

## 4. Run the checks and print the summary

In [4]:
def summarise(flags, total):
    """Counts per transaction: a row with two flags is counted once.
    'Error' = at least one Error flag; 'Review' = only Review flags."""
    flagged_rows = set(flags["Row"])
    error_rows = set(flags.loc[flags["Severity"] == "Error", "Row"])
    return {"Total transactions": total,
            "Flagged": len(flagged_rows),
            "Error": len(error_rows),
            "Review": len(flagged_rows - error_rows)}


def print_summary(flags, total):
    s = summarise(flags, total)
    print(f"{s['Flagged']} of {s['Total transactions']} transactions flagged "
          f"({s['Error']} Error, {s['Review']} Review)")


flags = find_anomalies(transactions, price_list)
print_summary(flags, len(transactions))
print()
print(flags.to_string(index=False, na_rep="(blank)"))

14 of 44 transactions flagged (9 Error, 5 Review)

 Row TransactionID CustomerID                 Item                    Rule                                 Reason Severity
   6         T0005      C1004           Chai Latte     Suspected duplicate                        0 s after T0004    Error
  10         T0009      C1007           Flat White           High quantity  Quantity 24 is above normal range 1-5   Review
  12         T0011      C1009           Cappuccino             Wrong price       Charged $55.00, list price $5.20    Error
  16         T0015      C1011         Chicken Wrap     Suspected duplicate                       33 s after T0014    Error
  19         T0018    (blank)            Croissant           Missing field                   Customer ID is blank   Review
  21         T0020      C1013            Croissant           High quantity Quantity 120 is above normal range 1-5   Review
  23         T0022      C1014     Blueberry Muffin             Wrong price        Charge

## 5. Tests

Each test builds a small set of transactions (or uses the real file) and compares the flags with what the rules
say should happen. These results go in the Problem B test table in the report.

In [5]:
def make_transactions(records):
    """Build a test table. Each record: (TransactionID, 'HH:MM:SS' on 7 Sep 2026, CustomerID, Item, Qty, UnitPrice)."""
    rows = []
    for tid, time, cust, item, qty, price in records:
        rows.append({"TransactionID": tid,
                     "Timestamp": pd.Timestamp(f"2026-09-07 {time}") if time else None,
                     "CustomerID": cust, "Item": item, "Quantity": qty,
                     "UnitPrice (AUD)": price,
                     "LineTotal (AUD)": None if qty is None or price is None else qty * price,
                     "PaymentMethod": "Card"})
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.insert(0, "Row", df.index + 2)
    return df


def rules_for(flags, tid):
    return sorted(flags.loc[flags["TransactionID"] == tid, "Rule"])


test_prices = pd.DataFrame({"Item": ["Flat White", "Croissant"], "List Price (AUD)": [5.20, 4.90]})
results = []

def check(test_id, description, expected, actual):
    results.append({"ID": test_id, "Test case": description,
                    "Expected": expected, "Actual": actual,
                    "Result": "Pass" if expected == actual else "FAIL"})


# B1 - typical: the real workbook
f = find_anomalies(transactions, price_list)
s = summarise(f, len(transactions))
check("B1", "Typical: full workbook (44 records)", "14 flagged (9 Error, 5 Review)",
      f"{s['Flagged']} flagged ({s['Error']} Error, {s['Review']} Review)")

# B2 - edge: exactly 60 s apart is flagged, 61 s is not
f = find_anomalies(make_transactions([
    ("A1", "09:00:00", "C1", "Flat White", 1, 5.20),
    ("A2", "09:01:00", "C1", "Flat White", 1, 5.20),     # 60 s after A1
    ("A3", "10:00:00", "C2", "Flat White", 1, 5.20),
    ("A4", "10:01:01", "C2", "Flat White", 1, 5.20)]),   # 61 s after A3
    test_prices)
a2, a4 = rules_for(f, "A2"), rules_for(f, "A4")
check("B2", "Edge: same customer + item at exactly 60 s and at 61 s", "A2 flagged, A4 not",
      "A2 flagged, A4 not" if a2 == ["Suspected duplicate"] and a4 == [] else f"A2: {a2}, A4: {a4}")

# B3 - same item, different customers, 39 s apart (like T0042/T0043) is NOT a duplicate
f = find_anomalies(make_transactions([
    ("B1", "14:05:51", "C1", "Croissant", 2, 4.90),
    ("B2", "14:06:30", "C2", "Croissant", 2, 4.90)]), test_prices)
check("B3", "Same item, different customers, 39 s apart", "no flags", "no flags" if f.empty else list(f["Rule"]))

# B4 - edge: blank Customer ID is Review and the other checks still run
f = find_anomalies(make_transactions([
    ("D1", "11:00:00", None, "Croissant", 8, 4.90),
    ("D2", "11:00:10", None, "Croissant", 1, 4.90)]), test_prices)
check("B4", "Edge: blank Customer ID, qty 8, repeat 10 s later",
      "D1: High quantity + Missing field; D2: Missing field",
      f"D1: {' + '.join(rules_for(f, 'D1'))}; D2: {' + '.join(rules_for(f, 'D2'))}")

# B5 - capitals and extra spaces in the item name are ignored
f = find_anomalies(make_transactions([("E1", "12:00:00", "C3", "  flat WHITE ", 1, 5.20)]), test_prices)
check("B5", "Item typed '  flat WHITE ' at the right price", "no flags", "no flags" if f.empty else list(f["Rule"]))

# B6 - unknown item is flagged and its price check is skipped
f = find_anomalies(make_transactions([("F1", "12:00:00", "C4", "Capuccino", 1, 99.00)]), test_prices)
check("B6", "Unknown item 'Capuccino' charged $99.00", "['Unknown item']", str(rules_for(f, "F1")))

# B7 - edge: quantity 0 is invalid
f = find_anomalies(make_transactions([("G1", "12:00:00", "C5", "Croissant", 0, 4.90)]), test_prices)
check("B7", "Edge: quantity 0", "['Invalid quantity']", str(rules_for(f, "G1")))

# B8 - identical timestamps: the later spreadsheet row is flagged (like T0004/T0005)
f = find_anomalies(make_transactions([
    ("H1", "07:31:52", "C6", "Flat White", 1, 5.20),
    ("H2", "07:31:52", "C6", "Flat White", 1, 5.20)]), test_prices)
check("B8", "Edge: identical timestamps", "H2 flagged (0 s after H1)",
      "H2 flagged (0 s after H1)" if list(f["Reason"]) == ["0 s after H1"] and list(f["TransactionID"]) == ["H2"]
      else list(zip(f["TransactionID"], f["Reason"])))

# B9 - a Transaction ID reused hours apart IS flagged, as Review (T0038 in the real file; T0041 is missing)
f = find_anomalies(transactions, price_list)
hit = f[f["Rule"] == "Repeated transaction ID"]
check("B9", "Repeated ID T0038, 3 h 9 min apart", "Row 42 flagged, Review",
      f"Row {hit['Row'].iloc[0]} flagged, {hit['Severity'].iloc[0]}" if len(hit) == 1 else str(list(zip(hit["Row"], hit["Severity"]))))

# B10 - line total that doesn't match quantity x unit price
t = make_transactions([("J1", "12:00:00", "C7", "Croissant", 2, 4.90)])
t.loc[0, "LineTotal (AUD)"] = 4.90                       # should be 9.80
f = find_anomalies(t, test_prices)
check("B10", "Line total $4.90 for 2 x $4.90", "['Wrong line total']", str(rules_for(f, "J1")))

# B11 - text in a number column is flagged instead of crashing
t = make_transactions([("K1", "12:00:00", "C8", "Croissant", 2, 4.90)])
t["Quantity"] = t["Quantity"].astype(object)
t.loc[0, "Quantity"] = "two"
f = find_anomalies(t, test_prices)
check("B11", "Quantity typed as 'two'", "['Not a number']", str(rules_for(f, "K1")))

# B12 - spaces and capitals in Customer IDs don't hide a duplicate
f = find_anomalies(make_transactions([
    ("L1", "13:00:00", "C9", "Flat White", 1, 5.20),
    ("L2", "13:00:20", " c9 ", "Flat White", 1, 5.20)]), test_prices)
check("B12", "Customer ID 'C9' then ' c9 ', 20 s apart", "['Suspected duplicate']", str(rules_for(f, "L2")))

# B13 - the same Transaction ID used twice within 60 s is an Error (a double entry), not just Review
f = find_anomalies(make_transactions([
    ("M1", "14:00:00", "C10", "Flat White", 1, 5.20),
    ("M1", "14:00:20", "C11", "Croissant", 1, 4.90)]), test_prices)
check("B13", "Edge: same ID 'M1' used twice, 20 s apart", "2nd record: Repeated transaction ID, Error",
      f"2nd record: {f['Rule'].iloc[0]}, {f['Severity'].iloc[0]}" if list(f["Row"]) == [3] else str(list(zip(f["Row"], f["Rule"], f["Severity"]))))

# B14 - text in the Timestamp column is flagged instead of crashing, and skips the time-gap checks
t = make_transactions([("N1", "15:00:00", "C12", "Flat White", 1, 5.20),
                       ("N2", "15:00:10", "C12", "Flat White", 1, 5.20)])
t["Timestamp"] = t["Timestamp"].astype(object)
t.loc[1, "Timestamp"] = "7/9 morning"
f = find_anomalies(t, test_prices)
check("B14", "Timestamp typed as '7/9 morning'", "N1: no flags; N2: ['Not a date']",
      f"N1: {'no flags' if not rules_for(f, 'N1') else rules_for(f, 'N1')}; N2: {rules_for(f, 'N2')}")

test_table = pd.DataFrame(results)
print(test_table.to_string(index=False))
print(f"\n{(test_table['Result'] == 'Pass').sum()} of {len(test_table)} tests passed")

 ID                                              Test case                                             Expected                                               Actual Result
 B1                    Typical: full workbook (44 records)                       14 flagged (9 Error, 5 Review)                       14 flagged (9 Error, 5 Review)   Pass
 B2 Edge: same customer + item at exactly 60 s and at 61 s                                   A2 flagged, A4 not                                   A2 flagged, A4 not   Pass
 B3             Same item, different customers, 39 s apart                                             no flags                                             no flags   Pass
 B4      Edge: blank Customer ID, qty 8, repeat 10 s later D1: High quantity + Missing field; D2: Missing field D1: High quantity + Missing field; D2: Missing field   Pass
 B5          Item typed '  flat WHITE ' at the right price                                             no flags                             

## 6. Interactive menu and owner review mode

Run the next two cells to use the menu. The owner's decisions are kept per transaction and saved to
`ProblemB_Flags.xlsx` only through menu option 5.

In [6]:
def ask_until_valid(prompt, valid):
    """Keep asking until the answer is one of `valid` (single letters/numbers, capitals ignored)."""
    while True:
        answer = input(prompt).strip().upper()
        if answer in valid:
            return answer
        print(f"Please type one of: {', '.join(valid)}")


def error_rows(flags):
    return set(flags.loc[flags["Severity"] == "Error", "Row"])


def rows_in_group(flags, group):
    """'A' = all flagged rows, 'E' = rows with at least one Error, 'R' = rows with Review flags only."""
    all_rows = set(flags["Row"])
    if group == "E":
        return error_rows(flags)
    if group == "R":
        return all_rows - error_rows(flags)
    return all_rows


def show_flags(flags, decisions, rows=None):
    shown = flags if rows is None else flags[flags["Row"].isin(rows)]
    if shown.empty:
        print("No flags to show.")
        return
    shown = shown.assign(**{"Owner decision": shown["Row"].map(decisions)})
    print(shown.to_string(index=False, na_rep="(blank)"))


def show_value(value):
    return "(blank)" if is_blank(value) else value


def save_to_excel(flags, decisions, transactions, output_file):
    """Sheet 'Flags': one line per flag, decision repeated per transaction. Sheet 'Summary': counts."""
    flag_sheet = flags.assign(**{"Owner decision": flags["Row"].map(decisions)})
    counts = summarise(flags, len(transactions))
    decided = pd.Series(decisions).value_counts()
    for label in ["Confirmed error", "Not an error", "Not reviewed"]:
        counts[label] = int(decided.get(label, 0))
    summary_sheet = pd.DataFrame(list(counts.items()), columns=["Measure", "Count"])
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        flag_sheet.to_excel(writer, sheet_name="Flags", index=False)
        summary_sheet.to_excel(writer, sheet_name="Summary", index=False)


def review_flags(transactions, flags, decisions):
    """Owner review mode. Returns True if any decision changed (so the menu knows there is something to save)."""
    group = ask_until_valid("Review which? A = All, E = Errors only, R = Review only: ", ["A", "E", "R"])
    to_review = sorted(r for r in rows_in_group(flags, group) if decisions[r] == "Not reviewed")
    if not to_review:
        print("Nothing left to review in this group.")
        return False

    changed = False
    for number, row_no in enumerate(to_review, start=1):
        record = transactions[transactions["Row"] == row_no].iloc[0]
        price = to_number(record["UnitPrice (AUD)"])          # None if blank or text
        print(f"\n[{number} of {len(to_review)}] Row {row_no}: {show_value(record['TransactionID'])} | "
              f"{show_value(record['Timestamp'])} | Customer {show_value(record['CustomerID'])} | "
              f"{show_value(record['Item'])} | Qty {show_value(record['Quantity'])} | "
              f"Unit price {show_value(record['UnitPrice (AUD)']) if price is None else money(price)}")
        for _, flag in flags[flags["Row"] == row_no].iterrows():
            print(f"   - {flag['Severity']}: {flag['Rule']} ({flag['Reason']})")

        answer = ask_until_valid("C = confirm error, N = not an error, S = skip, Q = quit: ",
                                 ["C", "N", "S", "Q"])
        if answer == "Q":
            print("Review stopped. Decisions made so far are kept.")
            return changed
        if answer == "C":
            decisions[row_no] = "Confirmed error"
            changed = True
        elif answer == "N":
            decisions[row_no] = "Not an error"
            changed = True                              # S = stays "Not reviewed"
    print("\nReview finished.")
    return changed


def run_menu(transactions, flags, decisions, output_file=OUTPUT_FILE):
    unsaved = False
    finished = False
    while not finished:
        print("\n1) Show all flags   2) Show Errors or Review only   3) Show one rule")
        print("4) Look up a transaction ID   5) Save flags to Excel   6) Review flags   7) Quit")
        choice = input("Choose 1-7: ").strip()

        if choice == "1":
            show_flags(flags, decisions)
        elif choice == "2":
            group = ask_until_valid("E = Errors, R = Review: ", ["E", "R"])
            show_flags(flags, decisions, rows_in_group(flags, group))
        elif choice == "3":
            rules = sorted(flags["Rule"].unique())
            if not rules:
                print("No flags to show.")
                continue
            for n, rule in enumerate(rules, start=1):
                print(f"   {n}) {rule}")
            pick = ask_until_valid(f"Rule number (1-{len(rules)}): ",
                                   [str(n) for n in range(1, len(rules) + 1)])
            show_flags(flags[flags["Rule"] == rules[int(pick) - 1]], decisions)
        elif choice == "4":
            tid = input("Transaction ID: ").strip().upper()            # 't0038 ' finds T0038
            ids = transactions["TransactionID"].astype(str).str.strip().str.upper()
            matches = transactions[ids == tid]                         # T0038 -> both rows
            if matches.empty:
                print(f"No transaction with ID {tid}")
            for row_no in matches["Row"]:
                if row_no in set(flags["Row"]):
                    show_flags(flags, decisions, {row_no})
                else:
                    print(f"Row {row_no}: {tid} - no flags (OK)")
        elif choice == "5":
            save_to_excel(flags, decisions, transactions, output_file)
            unsaved = False
            print(f"Saved to {output_file}")
        elif choice == "6":
            if review_flags(transactions, flags, decisions):
                unsaved = True
        elif choice == "7":
            if unsaved:
                answer = ask_until_valid("You have unsaved decisions. Save first? (Y/N): ", ["Y", "N"])
                if answer == "Y":
                    save_to_excel(flags, decisions, transactions, output_file)
                    print(f"Saved to {output_file}")
            finished = True
        else:
            print("Please enter a number from 1 to 7")
    print("Goodbye")

In [7]:
# Start the menu (this cell waits for typed input)
decisions = {row_no: "Not reviewed" for row_no in sorted(set(flags["Row"]))}
print_summary(flags, len(transactions))
run_menu(transactions, flags, decisions)

14 of 44 transactions flagged (9 Error, 5 Review)

1) Show all flags   2) Show Errors or Review only   3) Show one rule
4) Look up a transaction ID   5) Save flags to Excel   6) Review flags   7) Quit
Row 39: T0038 - no flags (OK)
 Row TransactionID CustomerID               Item                    Rule                               Reason Severity Owner decision
  42         T0038      C1009 Fresh Orange Juice Repeated transaction ID Same ID as row 39, 3 h 9 min earlier   Review   Not reviewed

1) Show all flags   2) Show Errors or Review only   3) Show one rule
4) Look up a transaction ID   5) Save flags to Excel   6) Review flags   7) Quit
Goodbye
